# View and cluster structure: the method

This chapter describes the `views` detector in full: the model it fits, the
scores it optimizes, how the search works, what each finding means, and where
the method's limits are. The companion chapter walks through examples; this
one is the reference.

**The idea in one paragraph.** Most profilers describe columns one at a time,
or pairs of columns. The views detector asks a different question: do some
columns carve the rows into the *same groups*? If temperature and pressure
both shift at the same row, they belong in one *view* — a block of columns
that move together. Within a view, rows split into *clusters*, each with its
own per-column distribution. Columns in different views are treated as
independent. That two-level partition — columns into views, rows into
clusters — is the whole model.

**The honesty line, up front.** Every views finding carries this line:
"Greedy deterministic partition; scores are marginal-likelihood values, not
posterior probabilities." The partition is a point estimate found by greedy
search, not a posterior sample. Scores are marginal likelihoods under stated
models, not probabilities of dependence. Nothing in this chapter claims
otherwise.

**Enabling.** Views is opt-in. It is excluded from the default detector set
and runs only when explicitly requested, with `detectors=["views"]`. It runs
at the standard and deep budgets; the quick tier skips it entirely.

## The model

A view is a set of columns modeled together. Within a view, rows are
partitioned into clusters. Each (cluster, column) pair has its own component
model: a Gaussian for continuous columns, a multinomial for categorical
ones. Columns are independent *given* the cluster — the model captures
columns whose distributions shift together across row groups, not
correlations within a group.

Two consequences worth knowing before anything else:

- A view with one column is the absence of detected dependence, not a
  statement. Singleton views emit no findings.
- The partition says nothing about row order. Rows are treated as an
  unordered set within a view. Time order, gaps, and cadence stay with the
  dates pass.

## Continuous columns: standardization, then a Gaussian model

Continuous columns are integers, floats, dates (days since epoch), and
datetimes (microseconds since epoch, naive). Before scoring, each column's
values are standardized on the sample:

```
z = (x - median) / IQR
```

with quartiles by linear interpolation (type 7) over the column's non-null
sample values. Median/IQR scaling is load-bearing, not cosmetic: mean/sd
scaling lets a few extreme spikes inflate the scale so far that a constant
block becomes indistinguishable from varied rows. Median/IQR keeps a
constant block at z = 0 and the varied rows spread around it. Fallbacks, in
order: if IQR is 0, use mean/sd; if that degenerates too, every standardized
value is 0.0.

Standardization is an empirical-Bayes-flavored step, stated openly: the
score is a marginal on the standardized scale, comparable across columns and
units, and absolute-scale information is lost. Anomaly rankings are
unaffected, because standardization is a positive linear map.

Each standardized column in each cluster is modeled as Gaussian with a
Normal-Inverse-Gamma prior, fixed for all continuous columns:
NIG(mu0 = 0, kappa0 = 1, alpha0 = 2, beta0 = 1). One pseudo-observation's
worth of prior mean, a finite prior mean for the variance, and a prior
variance scale of one standardized unit. No per-column tuning.

For values x_1..x_n with sample mean xbar and sum of squared deviations
S = sum (x_i - xbar)^2, the log marginal likelihood is:

```
kappa_n = kappa0 + n
alpha_n = alpha0 + n / 2
beta_n  = beta0 + S / 2 + kappa0 * n * (xbar - mu0)^2 / (2 * kappa_n)
log p(x) = lgamma(alpha_n) - lgamma(alpha0)
         + alpha0 * log(beta0) - alpha_n * log(beta_n)
         + (1 / 2) * (log(kappa0) - log(kappa_n))
         - (n / 2) * log(2 * pi)
```

All degenerate cases are finite: a single value, a constant column (S = 0),
and a cluster with no non-null values for the column (which contributes 0
to that column's sum — a column is scored only on rows where it is
observed).

The posterior predictive — used for anomaly scores — is a Student-t with
nu = 2 * alpha_n, location mu_n = (kappa0 * mu0 + n * xbar) / kappa_n, and
scale^2 = beta_n * (kappa_n + 1) / (alpha_n * kappa_n):

```
log p(x_new) = lgamma((nu+1)/2) - lgamma(nu/2)
             - (1/2) * log(nu * pi * scale^2)
             - ((nu+1)/2) * log(1 + (x_new - mu_n)^2 / (nu * scale^2))
```

A consistency property ties the two together, and the implementation is
tested on it: the predictive for a new value must exactly equal the
marginal with the value included minus the marginal without it.

## Categorical columns: capped levels, pooled tail, Dirichlet model

Categorical columns are strings and booleans. Levels are the distinct
non-null values on the sample, capped at 32: the 32 most frequent levels
(ties by level name ascending) keep their identity, and every other value
pools into one explicit `other` level. The pooled level is a real level in
the model, not a deletion — but rare-level structure inside the tail is
lost, and that loss is stated in the finding's metrics. If the literal
string "other" is itself a kept level, the pooled tail takes a distinct
name so the two can never collide.

The prior is a symmetric Dirichlet with concentration 1 per level
(Laplace smoothing). For counts c_1..c_K (K levels including the pooled
tail) and n = sum c_k:

```
log p = lgamma(K * alpha) - lgamma(n + K * alpha)
      + sum_k [lgamma(c_k + alpha) - lgamma(alpha)]
```

The predictive probability of level k is (c_k + alpha) / (n + K * alpha).

Dates and datetimes are *not* treated as categorical: they go through the
continuous family above, so order and spacing are preserved and regime
shifts in time appear as distribution shifts. Calendar structure
(day-of-week, seasonality, business calendars) is not modeled; a column
whose only structure is calendrical will not cluster meaningfully, and
the pass reports that as no improving split rather than inventing one.

## Nulls and ineligible columns

Nulls are excluded from component scores, for every type, uniformly. Each
(view, cluster, column) cell records its null count in evidence metrics, so
a cluster that is mostly null on some column is visible, not hidden. Nulls
are missingness, already covered by column-profile missingness findings;
scoring nulls as a category would let missingness patterns drive views and
clusters.

Ineligible columns — all-null columns and opaque dtypes — are recorded in
evidence metrics and in the coverage detail, not silently dropped. Only the
first 64 eligible columns in table order are considered; the rest are
named as dropped, so the exclusion is visible even when no views findings
are emitted.

## The score and the complexity penalty

For a partition into views V, where view v has clusters C(v):

```
score = sum_v [ sum_{c in C(v)} sum_{col in v} log-marginal(col values in c)
                - gamma * log(n_sample) * |C(v)| ]
        - gamma * log(n_sample) * |V|
```

with gamma = 2.0 by default (recorded in metrics) and n_sample the full
sample row count. Each new cluster must earn its keep by improving fit by
more than a small multiple of log n — the same order as a BIC penalty for a
handful of parameters — and each new view likewise. This is deliberately
not a Dirichlet-process prior: no concentration parameters to check, one
tunable constant, and the penalty is visible in every reported score.
Sensitivity to gamma is untested.

A merge of views v1, v2 into m changes the score by:

```
delta = score(m) - score(v1) - score(v2) + gamma * log(n_sample)
```

(the last term because the view count drops by one). A merge is accepted
iff delta > 0, computed on the search sample. Additionally, a merged view
with a single cluster is never accepted: it carries no partition evidence
and would merge on penalty savings alone. Two constant columns, for
example, gain exactly 2 * gamma * log(n) — 24.9 nats at 500 rows — with
zero marginal improvement, because the merged model is literally the
identical product model. The guard is deliberately narrower than a
marginals-only gate: genuine aligned merges also have zero marginal gain
(the joint clusters equal the singleton clusters), so a marginals-only
gate would kill the real cases along with the vacuous ones.

## How the search works

The search is deterministic greedy, in four stages. No randomness, no
wall-clock dependence: the same table always yields the same report,
byte for byte.

**Phase B: clustering within a view.** Divisive median splits. Start with
all search rows in one cluster. Repeatedly: for each cluster, for each
column in the view, consider splitting the cluster's rows at the
distinct-value boundary nearest the median position of that column's
non-null values within the cluster. For categorical columns the "values"
are level ids under the frequency-then-name encoding order, so the split
bipartitions levels by that order. Rows null on the split column go to the
majority side (ties to the lower-index side). The split gain is the
marginal of the left half plus the marginal of the right half minus the
marginal of the whole, summed over *all* columns in the view — the split
is evaluated on the view's joint score, not just the split column — minus
gamma * log(n_search) for the new cluster. Accept the best split iff its
gain is positive. Tie-break order, applied strictly: highest gain, then
lowest cluster index, then lowest column index. Stop when no split gains
or 8 clusters are reached.

The boundary rule matters: splitting at the median *value* degenerates
whenever the median equals the cluster maximum (left side = every row,
right side empty), which happens routinely for discrete-valued columns.
A singleton column then cannot split at all while a pair view can split
around it, manufacturing a spurious merge. Splitting at distinct-value
boundaries fixes this.

**Phase A: view discovery.** First, run Phase B on each eligible column
alone and record each singleton view's score. Then, for every column pair,
run Phase B on the two-column view and compute the pair gain (pair score
minus both singleton scores); sort pairs by gain descending, ties by
lowest column-index pair lexicographically. Then the merge loop, in
pre-screen order: for each pair whose columns are in different views,
compute the merged view's Phase B clustering and score on the search
sample, and accept iff delta > 0. Stop after 64 merge attempts or when
the pair list is exhausted.

**Assignment.** Every full-sample row is assigned to the cluster
maximizing its total log predictive density across the view's columns
(nulls contribute 0). Ties go to the lowest cluster index. Scoring is
leave-one-out: a row is scored under its own cluster's statistics with
itself removed, so outliers cannot partly explain themselves. Rows that
did not contribute to the statistics (non-search rows on the first pass)
score under the full statistics. Assignment runs twice: search statistics
assign, full statistics from that assignment reassign once, and final
statistics come from the second assignment.

**Final scoring.** All marginals are recomputed on the full sample with
the final assignments. Reported scores are full-sample scores; search
decisions are search-sample decisions, and that distinction is recorded
in metrics. Merge-gain claims state the search-sample scale explicitly.

**Budget and deadline.** Split evaluation stops when cell visits exceed
3e8 (configurable), records the hit, and keeps the current partition;
a budget-truncated run reports as capped, with any columns dropped
mid-singleton by the budget listed by name. The pipeline's deadline
pre-check skips the pass entirely if the deadline fired before it starts,
with a coverage note. Runtime is never recorded in evidence metrics:
wall-clock values in the report would break byte-identical determinism.

## The findings: dependence blocks

Category `associations`, type `statistical`, subject the view's columns.
Claim template:

> Columns {a, b, ...} form a dependence block under the greedy view
> partition (merge gain {g} nats over separate views on the search
> sample; {honesty}).

One finding per multi-column view. A dependence block means the columns
share a row partition — their distributions shift together across the
same row groups. It does not mean the columns are correlated within a
group: the model assumes independence given the cluster, so within-cluster
correlation is invisible to this pass by construction.

Detector priority 0.60. Note the reporting caveat: the report selector
overwrites detector priorities with its own score, which demotes
associations (no correlation keys are emitted, so strength reads 0).
The emitted order is therefore not the detector's order; fixing the
detector/selector metric contract is open work.

## The findings: regime splits

Category `changes`, type `residual_level`, subject the view's columns.
Claim template:

> Rows split into {k} clusters; headline column {c} shifts by {s}
> between clusters ({honesty}).

One finding per view with at least 2 clusters, describing the largest
between-cluster shift. The shift is in per-column median/IQR
(standardized) units — not raw units — and the headline compares
continuous z-shifts against categorical total-variation distances on one
scale, so a categorical column effectively never headlines a mixed view.
The specification records the reference model ("single-component per
column"), the headline column, and per-cluster row ranges, sizes, and
headline-column means, so the statement is interpretable without this
chapter.

Detector priority 0.70, the highest of the views kinds.

## The findings: degenerate clusters

Category `concentration`, type `modal_share`, subject the column. Claim
template:

> {n} of {m} rows in cluster {i} (rows {r1}-{r2}) hold {column} at {v}
> ({share}% of the cluster; {honesty}).

One statement per column, listing all modal clusters when there are
several. A cluster qualifies when it has at least 30 non-null
observations of the column and one exact value (continuous) or level
(categorical) holding at least 90% of the cluster's non-null rows.
Purity is deliberately not required: greedy search rarely isolates a
constant block exactly — it usually lands as the dominant share of a
slightly mixed cluster, and the 90% share is the honest statement of
that structure.

This is how a stuck run surfaces: 600 rows at constant 21.5 have *high*
predictive density under their own degenerate cluster, so the anomaly
score below never flags them. The degenerate finding is the one that
names them. One score does not do both jobs, and the design does not
pretend otherwise.

Detector priority 0.60.

## The findings: spike cells

Category `concentration`, type `spike_cells`, subject the column. Claim
template:

> {n} cells in {column} have log predictive density below {threshold}
> nats under their cluster (lowest {v} at row {r}; {honesty}).

For each column, cells with log predictive density below (column median
log-density minus 10 nats) are reported, lowest first, capped at 20 per
column. Ten nats is a factor of about 22000 in density against a typical
cell — a strong-evidence line, not a mild-deviation flag. Reported per
finding: count, the cell values, and original row indices (sample rows
map back through the stratified sample to table rows). The threshold and
the cap are recorded in metrics.

Small-cluster rule, load-bearing: cells in clusters smaller than 30
sample rows are scored under the view's *largest* cluster, not their
own. Without this, a handful of identical spikes forms its own cluster
and looks perfectly typical under it, making the spikes invisible to
exactly the score meant to catch them. The trade-off is stated: a
genuine small regime (under 30 sample rows) is reported as outlying
cells rather than as a regime. Cells scored under their own cluster use
leave-one-out statistics; cells scored under the largest cluster are not
in its statistics and need no adjustment.

Detector priority 0.50.

## Configuration

All bounds are count-based, so the pass introduces no nondeterminism.
Defaults, with what each knob does:

- `row_cap`: sample rows for final scoring. Standard 20,000; deep 50,000.
- `search_rows`: rows used for search decisions (splits, merges). 2000.
- `max_columns`: eligible columns considered, in table order. 64.
- `max_clusters`: clusters per view. 8.
- `tried_merges_cap`: merge attempts in the merge loop. 64.
- `nominal_max_levels`: categorical levels kept before pooling the tail. 32.
- `gamma`: complexity penalty multiplier, nats per cluster/view times
  log n. 2.0.
- `anomaly_threshold_nats`: spike threshold below the column median
  log-density. 10.0.
- `anomaly_cap`: anomaly cells reported per column. 20.
- `min_degenerate_rows`: minimum non-null observations for a degenerate
  finding, and the small-cluster threshold for spike scoring. 30.
- `work_budget`: cell-visit budget for split evaluation. 3e8.

Evidence metrics for every finding include: `gamma`, `n`, `search_rows`,
`max_columns`, `max_clusters`, `anomaly_threshold_nats`, `anomaly_cap`,
whole-sample per-column null counts, dropped-column names, and the
full-sample partition score (`view_score_nats`) plus merge gain
(`merge_gain_nats`).

## Limits, stated honestly

- **Noise splits.** The marginal-likelihood criterion splits pure-noise
  columns too: a unimodal sample cut into tighter pieces gains marginal
  likelihood faster than the gamma = 2.0 penalty removes it. Pure noise
  (2000 iid standard normals) splits into 6 clusters with a headline
  shift of 2.51, while real 0.5-sigma and 1-sigma regimes land at 2.44
  and 2.54 in the same units — statistically indistinguishable. No
  threshold separates them. Cluster counts and regime findings on noisy
  columns are therefore not, on their own, evidence of regimes. The fix
  direction is a null-calibrated gate (compare each split's gain against
  the gain expected from splitting pure noise of the same size) or
  reporting regime gaps in sigma units; neither is implemented.
- **Background-rule cliff.** Cells in clusters below 30 rows score under
  the largest cluster, so one row crossing 30 flips a constant block
  from spike findings to a degenerate finding. The cliff is documented,
  not smoothed: a smooth rule needs a design decision, and any candidate
  must not re-hide identical-spike clusters.
- **Pooled tail disables spike detection.** Rare levels pool into one
  Dirichlet cell, so after roughly 500 tail rows an unseen rare level
  scores about -0.06 nats — structurally invisible as a spike. Tail-only
  structure (rare_a -> rare_b versus rare_c -> rare_d) collapses to one
  cell and is hidden. A separate anomaly path for rare levels is open
  work.
- **Search-sample gap.** Split and merge decisions run on a stratified
  2000-row subsample, so regimes confined to under ~10% of rows (under
  ~200 search rows) may never split. Merge and split decisions that are
  right on the search sample can be wrong on the full sample; metrics
  record both sample sizes so the gap is visible.
- **Ledger contract.** The report selector overwrites detector
  priorities and rescores findings on keys the views pass does not emit,
  so the emitted finding order is not the detector's order (see the
  dependence-block section). Fixing the contract is open work.
- **Untested sensitivities.** Gamma = 2.0 is a chosen default, not a
  derived value; sensitivity to gamma is untested. The scale case (1M
  rows by 61 columns) has not been measured; no performance claim is
  made beyond the smoke bound below. Planted relationships in columns
  beyond position 64 cannot be detected under the 64-column cap at all.
- **What the model cannot see.** Within-cluster correlation, calendar
  structure in dates, and missingness-driven structure (nulls are
  excluded by design) are all invisible to this pass. Each has a sibling
  pass that owns it.

## Validation

Criteria fixed in advance, then checked on a 50,000-row sensor table
with a planted regime shift at row 25000, a 600-row stuck run at 21.5,
and 15 spikes at 999.9:

- **Regime boundary.** The temperature view recovers the shift with
  boundary 24997 (target 25000).
- **Spikes.** All 7 sampled spikes are flagged at the standard budget;
  all 15 at full scope. A pressure value (1022.15 at row 36854) not in
  the planted ground truth is also flagged — reported, not asserted.
- **Determinism.** Two runs produce byte-identical reports.
- **Runtime.** 0.9 seconds at the standard budget on the sensor table.
  The scale case is unmeasured (see Limits).
- **Dependence block.** The device change ties to its row partition:
  device_id and timestamp form one block (merge gain 30.4 nats), with
  one cluster covering rows 29406-49997 entirely at one device value.

The worked example below is checked, not asserted: the numbers come
from direct evaluation of the formulas above.

## Worked example

Continuous, NIG(0, 1, 2, 1). Raw values [19.0, 20.0, 21.0]: median 20.0,
quartiles (type 7) 19.5 and 20.5, IQR 1.0, standardized values [-1, 0, 1].
Then n = 3, xbar = 0, S = 2.0, kappa_n = 4, alpha_n = 3.5, beta_n = 2.0,
and:

```
log p = lgamma(3.5) - lgamma(2) + 0 - 3.5 * log(2.0)
        + 0.5 * (log 1 - log 4) - 1.5 * log(2 * pi)
      = 1.200974 - 2.426015 - 0.693147 - 2.756816
      = -4.67500
```

Categorical, Dirichlet(1). Counts [3, 1] over K = 2 levels:

```
log p = lgamma(2) - lgamma(6) + (lgamma(4) - lgamma(1))
        + (lgamma(2) - lgamma(1))
      = 0 - 4.78749 + 1.79176 + 0
      = -2.99573
```

Predictive check on the NIG cluster above (mu_n = 0, nu = 7,
scale^2 = 0.7143): log predictive density at a typical value (0) is
-0.7863; at a standardized spike (1200) it is -51.07. The ~50-nat gap
is why the 10-nat spike threshold separates spikes from typical cells
with wide margin.

For a walked-through demonstration on data, see the companion chapter
on view and cluster structure.